# Logistic Regression Baseline

Train the project’s primary classical baseline from the existing answer-level train, validation, and test CSVs. The text input is transformed with a word-level TF-IDF representation; the same transformed features and selected settings are suitable for comparison with the other classical classifiers.

The train/validation/test files were generated by the project’s question-grouped split notebook. Use validation macro-F1 to choose regularization strength, then refit the vectorizer and model on train plus validation. Keep the test set untouched until final evaluation.

Run `data_cleaning.ipynb` and `dataset_splitting.ipynb` first.


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)


## Load the saved split and check question isolation

The project’s expanded files use `text` for answer content and `target` for the numeric label (`0 = human`, `1 = ChatGPT`). Do not re-split answer rows: all answers belonging to one question must stay together.


In [ ]:
PROCESSED_DIR = Path("../data/processed")
train = pd.read_csv(PROCESSED_DIR / "train.csv")
validation = pd.read_csv(PROCESSED_DIR / "validation.csv")
test = pd.read_csv(PROCESSED_DIR / "test.csv")

required = {"id", "source", "text", "target"}
for name, frame in (("train", train), ("validation", validation), ("test", test)):
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"{name} split is missing columns: {sorted(missing)}")
    if frame["text"].isna().any() or frame["target"].isna().any():
        raise ValueError(f"{name} split contains missing text or target values")

train_ids = set(train["id"])
validation_ids = set(validation["id"])
test_ids = set(test["id"])
assert not (train_ids & validation_ids or train_ids & test_ids or validation_ids & test_ids)

for name, frame in (("Train", train), ("Validation", validation), ("Test", test)):
    print(f"{name}: {len(frame):,} answers; {frame['id'].nunique():,} questions")
    print(frame["target"].value_counts().sort_index().rename(index={0: "human", 1: "ChatGPT"}).to_dict())


## Create the shared TF-IDF representation

Use word unigrams and bigrams with sublinear term frequency. The vocabulary is learned from training text only during hyperparameter selection. `class_weight="balanced"` compensates for the project split’s unequal number of human and ChatGPT answers.


In [ ]:
VECTORIZER_CONFIG = {
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_features": 100_000,
    "sublinear_tf": True,
    "strip_accents": "unicode",
    "dtype": np.float32,
}
C_VALUES = [0.1, 1.0, 4.0, 10.0, 20.0, 50.0, 100.0]
RANDOM_STATE = 42

vectorizer = TfidfVectorizer(**VECTORIZER_CONFIG)
X_train = vectorizer.fit_transform(train["text"].astype(str))
X_validation = vectorizer.transform(validation["text"].astype(str))
y_train = train["target"].astype(int)
y_validation = validation["target"].astype(int)

validation_rows = []
for c_value in C_VALUES:
    candidate = LogisticRegression(
        C=c_value,
        class_weight="balanced",
        solver="lbfgs",
        max_iter=1000,
        random_state=RANDOM_STATE,
    )
    candidate.fit(X_train, y_train)
    predictions = candidate.predict(X_validation)
    validation_rows.append({
        "C": c_value,
        "accuracy": accuracy_score(y_validation, predictions),
        "macro_f1": f1_score(y_validation, predictions, average="macro"),
    })

validation_results = pd.DataFrame(validation_rows)
print(validation_results.round(4).to_string(index=False))
best_c = float(validation_results.sort_values(
    ["macro_f1", "accuracy"], ascending=False
).iloc[0]["C"])
print(f"Selected C={best_c:g} using validation macro-F1")


## Refit with the selected setting and evaluate the held-out test set

After selecting `C` on validation, fit the final vectorizer and classifier on train plus validation. Then evaluate once on the untouched test split. The output includes macro precision/recall/F1, per-class metrics, the confusion matrix, and a majority-class reference.


In [ ]:
train_validation = pd.concat([train, validation], ignore_index=True)
y_train_validation = train_validation["target"].astype(int)
final_vectorizer = TfidfVectorizer(**VECTORIZER_CONFIG)
X_train_validation = final_vectorizer.fit_transform(train_validation["text"].astype(str))
X_test = final_vectorizer.transform(test["text"].astype(str))
y_test = test["target"].astype(int)

model = LogisticRegression(
    C=best_c,
    class_weight="balanced",
    solver="lbfgs",
    max_iter=1000,
    random_state=RANDOM_STATE,
)
model.fit(X_train_validation, y_train_validation)
test_predictions = model.predict(X_test)

majority_prediction = int(y_train_validation.mode().iloc[0])
print(f"Majority-class test accuracy: {accuracy_score(y_test, np.full(len(y_test), majority_prediction)):.4f}")
print(f"Accuracy:        {accuracy_score(y_test, test_predictions):.4f}")
print(f"Macro precision: {precision_score(y_test, test_predictions, average='macro', zero_division=0):.4f}")
print(f"Macro recall:    {recall_score(y_test, test_predictions, average='macro', zero_division=0):.4f}")
print(f"Macro F1:        {f1_score(y_test, test_predictions, average='macro', zero_division=0):.4f}")
print("\nPer-class metrics:\n" + classification_report(
    y_test, test_predictions, labels=[0, 1], target_names=["human", "ChatGPT"], zero_division=0
))

cm = confusion_matrix(y_test, test_predictions, labels=[0, 1])
print("Confusion matrix (rows=true, columns=predicted; human, ChatGPT):\n", cm)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["human", "ChatGPT"]).plot(
    cmap="Blues", values_format="d"
)
plt.title("Logistic Regression: HC3 test set")
plt.tight_layout()
plt.show()


## Review behavior by domain and inspect influential vocabulary

Domain metrics help show where the overall score may hide uneven performance. The strongest positive and negative linear weights are descriptive model signals; they should not be interpreted as causal explanations.


In [ ]:
test_results = test[["id", "source", "text", "target"]].copy()
test_results["prediction"] = test_predictions
rows = []
for source, group in test_results.groupby("source", sort=True):
    rows.append({
        "source": source,
        "n_answers": len(group),
        "accuracy": accuracy_score(group["target"], group["prediction"]),
        "macro_f1": f1_score(group["target"], group["prediction"], average="macro", zero_division=0),
    })
print(pd.DataFrame(rows).round(4).to_string(index=False))

terms = np.asarray(final_vectorizer.get_feature_names_out())
weights = model.coef_[0]
print("\nTop words associated with ChatGPT:")
print(pd.DataFrame({"term": terms[np.argsort(weights)[-15:][::-1]],
                    "weight": np.sort(weights)[-15:][::-1]}).to_string(index=False))
print("\nTop words associated with human answers:")
print(pd.DataFrame({"term": terms[np.argsort(weights)[:15]],
                    "weight": np.sort(weights)[:15]}).to_string(index=False))


## Save the fitted baseline

This saves the fitted vectorizer and classifier together so later model-selection or demo work can load the exact baseline. The generated artifact stays under `data/processed/` and is not committed.


In [ ]:
model_path = PROCESSED_DIR / "logistic_regression.joblib"
joblib.dump({"vectorizer": final_vectorizer, "classifier": model,
             "config": {**VECTORIZER_CONFIG, "C": best_c,
                        "class_weight": "balanced", "solver": "lbfgs",
                        "max_iter": 1000, "random_state": RANDOM_STATE}},
            model_path)
print(f"Saved fitted baseline to {model_path}")
